In [ ]:
# I count connected components of kNN graph instead of runnning a ripser on it

In [ ]:
import sys
!{sys.executable} -m pip install dill --user

In [ ]:
# initialize and load the data
# Initialize the optimizer
optimizer = MonteCarloKNNOptimizer(
    data_root='${TDL_ROOT_DIR}/results/D1',
    true_b0=9,
    data_file='only_0.dill'
)

# Load the data
optimizer.load_data()

In [ ]:
# Run a quick test with fewer trials to make sure everything works
print("=== Quick Test (100 trials) ===")
successful_ks_test = optimizer.monte_carlo_k_optimization(
    n_trials=100,           # Small number for testing
    subset_fraction=0.25,   # Use 25% of data each time
    max_k=30               # Don't test very high k values
)

if successful_ks_test:
    print(f"\n✅ Quick test successful! Found {len(successful_ks_test)} valid k values")
    print(f"Sample k values: {successful_ks_test[:9]}")
else:
    print("❌ Quick test failed - check data loading")

# Analyze the quick test results
if successful_ks_test:
    mean_k_test, std_k_test = optimizer.analyze_k_distribution(successful_ks_test)
    print(f"Quick test mean k: {mean_k_test:.2f}")

In [ ]:
# Run the full Monte Carlo optimization
print("=== Full Monte Carlo Optimization ===")

successful_ks = optimizer.monte_carlo_k_optimization(
    n_trials=1000,          # Full number of trials
    subset_fraction=0.25,   # Use 25% of data each time
    max_k=50,              # Test up to k=50
)

print(f"\nFound {len(successful_ks)} successful k values out of 1000 trials")

In [ ]:
# Analyze the full results
if successful_ks:
    mean_k, std_k = optimizer.analyze_k_distribution(successful_ks)
    
    # Print summary statistics
    print(f"\n📊 SUMMARY RESULTS:")
    print(f"   Optimal k (mean): {mean_k:.2f}")
    print(f"   Standard deviation: {std_k:.2f}")
    print(f"   Success rate: {len(successful_ks)/1000*100:.1f}%")
else:
    print("❌ No successful k values found")

In [ ]:
# Find a specific subset that works with the mean k
if 'mean_k' in locals() and mean_k:
    print(f"Finding subset that works with k = {int(round(mean_k))}")
    
    best_subset = optimizer.find_best_subset_for_k(
        target_k=mean_k,
        n_attempts=100,
        subset_fraction=0.25
    )
    
    if best_subset is not None:
        print(f"\n🎯 FINAL RESULTS:")
        print(f"   Optimal k: {int(round(mean_k))}")
        print(f"   Subset size: {len(best_subset)}")
        print(f"   Data shape: {best_subset.shape}")
        print(f"   Ready for outlier removal and Ripser analysis!")
        
        # Verify it works
        k_verify = optimizer.find_k_for_subset(best_subset, max_k=int(round(mean_k))+1)
        print(f"   Verification: k={k_verify} gives B0={optimizer.true_b0} ✅")
    else:
        print("❌ Could not find suitable subset")

In [ ]:
# Save all results for later use
if 'best_subset' in locals() and best_subset is not None:
    results = {
        'optimal_k': int(round(mean_k)),
        'mean_k': mean_k,
        'std_k': std_k,
        'all_successful_ks': successful_ks,
        'best_subset': best_subset,
        'subset_shape': best_subset.shape,
        'data_fraction_used': 0.25,
        'true_b0': optimizer.true_b0
    }
    
    # Save to pickle file
    with open('only_0_monte_carlo_results.pkl', 'wb') as f:
        pickle.dump(results, f)
    
    print("💾 Results saved to: only0_monte_carlo_results.pkl")
    print("\nNext steps:")
    print("1. Remove outliers from best_subset")
    print("2. Run Ripser on cleaned data") 
    print("3. Track components through layers")
    
    # Also save just the subset as numpy array for easy loading
    np.save('only0_optimal_subset.npy', best_subset)
    print("📁 Subset saved to: only0_optimal_subset.npy")

In [ ]:
# Inspect the optimal subset
if 'best_subset' in locals() and best_subset is not None:
    plt.figure(figsize=(10, 6))
    
    # If 2D data, plot scatter
    if best_subset.shape[1] == 2:
        plt.subplot(1, 2, 1)
        plt.scatter(best_subset[:, 0], best_subset[:, 1], alpha=0.6, s=20)
        plt.title(f'Optimal Subset (k={int(round(mean_k))})')
        plt.xlabel('Feature 1')
        plt.ylabel('Feature 2')
        plt.grid(True, alpha=0.3)
    
    # Plot distribution of features
    plt.subplot(1, 2, 2)
    for i in range(min(best_subset.shape[1], 3)):  # Plot up to 3 features
        plt.hist(best_subset[:, i], alpha=0.5, label=f'Feature {i+1}', bins=20)
    plt.xlabel('Value')
    plt.ylabel('Frequency')
    plt.title('Feature Distributions')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print(f"Subset statistics:")
    print(f"  Shape: {best_subset.shape}")
    print(f"  Mean: {np.mean(best_subset, axis=0)}")
    print(f"  Std: {np.std(best_subset, axis=0)}")

In [ ]:
# Left plot: Several distinct groups of points
# Right plot: Multiple peaks in histograms
# → This explains why B0=9 (9 connected components)